# Gate 5 (04) — Overfit Training with Mandatory Gates

**Purpose:** Smoke-test the full training loop on 5-10 CMU ARCTIC utterances.

**Three mandatory gates:**
- **A. Denoising:** denoised_zc1 is closer to clean than noisy input (improvement > 5%)
- **B. Mean baseline:** model beats predicting training-set mean
- **C. Conditioning:** correct phones < wrong phones < shuffled phones

**Stop condition:** If ANY gate fails at the final check → abort.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
# 2. Setup paths and environment
import os, sys, json, time, subprocess, types, warnings, shutil
from pathlib import Path

warnings.simplefilter("ignore")

ACCENTEDGE_DIR = "/content/accentedge"
FA_CODEC_DIR   = "/content/FAcodec"
GATE_DIR       = "/content/gate5_artifacts"
DRIVE_BASE     = "/content/drive/MyDrive/accentedge/runs"
AUDIO_DIR      = "/content/overfit_audio"
CONFIG_PATH    = f"{ACCENTEDGE_DIR}/configs/phase1/overfit.yaml"
SAMPLE_RATE    = 24000

def run(cmd, desc="", check=True, timeout=120):
    print(f"\n>>> {desc or cmd[:80]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    out = r.stdout.strip()
    if out:
        print(out[:500])
    if check and r.returncode != 0:
        print(f"FAILED: {r.stderr[:500]}")
        raise RuntimeError(f"Command failed: {cmd}")
    return r

# ── GPU ──
r = run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", "GPU", check=False)
gpu_info = r.stdout.strip()
print(f"GPU: {gpu_info}")

# ── Git SHA ──
r = run("cd /content/accentedge && git rev-parse HEAD", "SHA", check=False)
git_sha = r.stdout.strip() or "unknown"
print(f"accentedge SHA: {git_sha}")

# ── Environment manifest ──
import torch
manifest = {
    "gpu_name": gpu_info,
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else "N/A",
    "cuda_available": torch.cuda.is_available(),
    "accentedge_git_sha": git_sha,
    "manifest_timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "gate": "gate5_overfit",
}
os.makedirs(GATE_DIR, exist_ok=True)
with open(f"{GATE_DIR}/environment.json", "w") as f:
    json.dump(manifest, f, indent=2)

In [ ]:
# 3. Clone repos (idempotent)
run("test -d /content/FAcodec || git clone https://github.com/Plachtaa/FAcodec.git /content/FAcodec",
    "clone FAcodec", check=False)
run("test -f /content/FAcodec/modules/__init__.py || touch /content/FAcodec/modules/__init__.py",
    "modules init", check=False)
run("test -d /content/accentedge || git clone --depth 1 https://github.com/yagami009/accentedge.git /content/accentedge",
    "clone accentedge", check=False)

In [ ]:
# 4. Install dependencies
!pip install -q numpy soundfile librosa scipy jiwer pyyaml einops \
    huggingface-hub phonemizer torchaudio transformers speechbrain \
    faster-whisper pytest pyworld munch plotly datasets
print("Dependencies installed")

In [ ]:
# 5. Path setup + mock audiotools
sys.path = [p for p in sys.path if "/content" not in p]
sys.path.insert(0, FA_CODEC_DIR)
sys.path.insert(0, f"{ACCENTEDGE_DIR}/src")
os.environ["PYTHONPATH"] = FA_CODEC_DIR + "/modules:" + os.environ.get("PYTHONPATH", "")
os.chdir(FA_CODEC_DIR)

def _make_mock(name):
    m = types.ModuleType(name)
    m.__path__ = []
    m.__package__ = name
    return m

mock_audio = _make_mock("audiotools")
mock_ml = _make_mock("audiotools.ml")
mock_ml.BaseModel = type("BaseModel", (), {"INTERN": [], "EXTERN": []})
mock_audio.ml = mock_ml
mock_audio.AudioSignal = type("AudioSignal", (), {})
mock_audio.STFTParams = type("STFTParams", (), {})
mock_core = _make_mock("audiotools.core")
mock_core.util = _make_mock("audiotools.core.util")
sys.modules["audiotools"] = mock_audio
sys.modules["audiotools.ml"] = mock_ml
sys.modules["audiotools.core"] = mock_core
sys.modules["audiotools.core.util"] = mock_core.util
print("Path setup complete")

In [ ]:
# 6. Download CMU ARCTIC training audio (5-10 utterances)
import torchaudio, torch, numpy as np

os.makedirs(AUDIO_DIR, exist_ok=True)

print("\n=== Downloading CMU ARCTIC ===")
arctic_root = "/tmp/cmu_arctic"
os.makedirs(arctic_root, exist_ok=True)

speakers = ["slt", "bdl"]
audio_paths = []
transcripts = {}

for spk in speakers:
    ds = torchaudio.datasets.CMU_ARCTIC(root=arctic_root, speaker=spk, download=True)
    n_take = min(4, len(ds))  # 4 per speaker = 8 total
    idxs = np.random.RandomState(42).choice(len(ds), size=n_take, replace=False)
    for k, idx in enumerate(idxs):
        wav, sr, transcript = ds[idx]
        fname = f"{AUDIO_DIR}/{spk}_{k:03d}.wav"
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        torchaudio.save(fname, wav, SAMPLE_RATE)
        key = f"{spk}_{k:03d}"
        audio_paths.append(fname)
        transcripts[key] = transcript
        print(f"  {fname}: {transcript[:60]}")

print(f"\nTotal training utterances: {len(audio_paths)}")

In [ ]:
# 7. Save transcripts JSON for training
transcripts_path = f"{AUDIO_DIR}/transcripts.json"
with open(transcripts_path, "w") as f:
    json.dump(transcripts, f, indent=2)
print(f"Transcripts saved: {transcripts_path}")
print(f"\nTranscripts preview:")
for k, v in list(transcripts.items())[:5]:
    print(f"  {k}: {v[:60]}")

In [ ]:
# 8. Train overfit model with mandatory gates
import json as _json
from accentedge.training.overfit import train_overfit

print(f"\n{'='*60}")
print(f"GATE 5 — OVERFIT TRAINING")
print(f"{'='*60}")
print(f"Audio dir:    {AUDIO_DIR}")
print(f"Utterances:   {len(audio_paths)}")
print(f"Config:       {CONFIG_PATH}")
print(f"Device:       cuda")
print(f"{'='*60}\n")

metrics = train_overfit(
    audio_dir=AUDIO_DIR,
    transcripts=transcripts,
    output_dir=GATE_DIR,
    model_d_model=256,
    model_nhead=4,
    model_num_layers=3,
    model_d_ff=512,
    model_phone_vocab_size=393,
    model_facodec_dim=8,
    num_timesteps=100,
    num_steps=500,
    learning_rate=1e-4,
    device="cuda",
    facodec_ckpt="Plachta/FAcodec",
    seed=42,
    zc2_loss_weight=0.5,
    checkpoint_every=100,
    gate_check_every=100,
    decode_steps=[500],
)

In [ ]:
# 9. Check final gate results
final_gates = metrics.get("final_gates", {})
print("\n" + "="*60)
print("GATE 5 — FINAL GATE EVALUATION")
print("="*60)

all_pass = True
for gate_name in ["denoising", "mean_baseline", "conditioning"]:
    g = final_gates.get(gate_name, {})
    passed = g.get("passed", False)
    msg = g.get("msg", "N/A")
    symbol = "PASS" if passed else "FAIL"
    print(f"  {symbol}: {gate_name}")
    print(f"         {msg}")
    if not passed:
        all_pass = False

print(f"\n{'='*60}")
print(f"  OVERALL: {'GATE 5 PASSED' if all_pass else 'GATE 5 FAILED'}")
print(f"{'='*60}")

if not all_pass:
    print("\nOne or more gates failed. Aborting pipeline.")
    raise RuntimeError("GATE 5 FAILED — aborting pipeline.")

In [ ]:
# 10. Save artifacts to Drive
import shutil

drive_out = f"{DRIVE_BASE}/{git_sha}/gate5"
os.makedirs(drive_out, exist_ok=True)

# Copy key artifacts
for fname in ["metrics.json", "zc1_stats.json", "environment.json"]:
    src = f"{GATE_DIR}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, f"{drive_out}/{fname}")

# Copy checkpoints
ckpt_dir = GATE_DIR
for f in os.listdir(ckpt_dir):
    if f.endswith(".pt"):
        shutil.copy2(f"{ckpt_dir}/{f}", f"{drive_out}/{f}")

# Copy WAVs
wav_src = f"{GATE_DIR}/generated_wavs"
if os.path.isdir(wav_src):
    wav_dst = f"{drive_out}/generated_wavs"
    os.makedirs(wav_dst, exist_ok=True)
    for f in os.listdir(wav_src):
        if f.endswith(".wav"):
            shutil.copy2(f"{wav_src}/{f}", f"{wav_dst}/{f}")

print(f"\nArtifacts saved to: {drive_out}")
print("Files:")
for f in sorted(os.listdir(drive_out)):
    fpath = f"{drive_out}/{f}"
    size = os.path.getsize(fpath) if os.path.isfile(fpath) else "DIR"
    print(f"  {f}: {size}")

In [ ]:
# 11. Gate 5 complete
print(f"\nGate 5 {'PASSED' if all_pass else 'FAILED'}.")
print(f"Results: {GATE_DIR}/metrics.json")